# Basic RAG : 문서 기반 답변 만들기

1. Indexing Phase
- 외부 문서를 읽고 검색하기 좋은 단위로 나눈다.
- 임베딩 벡터로 변환한 뒤 VectorStore에 저장한다.

2. Retrival & Generation Phase
- 임베딩 된 사용자 질문과 관련있는 문서를 검색한다.
- 검색된 문서를 prompt의 context로 넣는다.
- LLM이 context를 근거로 답변한다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
%pip install langchain-chroma langchain-text-splitters

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ------------ --------------------------- 7.6/23.5 MB 39.3 MB/s eta 0:00:01
   ------------------------------- -------- 18.6/23.5 MB 43.5 MB/s eta 0:00:01
   ---------------------------------------- 23.5/23.5 MB 39.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 22.5 MB/s  0:00:00
   ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
   ------------------------------- -------- 10.2/13.0 MB 58.3 MB/s eta 0:00:01
   ---------------------------------------- 13.0/13.0 MB 37.2 MB/s  0:00:00

   ----- ----------------------------------  3/24 [pybase64]
  Attempting uninstall: protobuf
   ----- ----------------------------------  3/24 [pybase64]
    Found existing installation: protobuf 7.34.1
   ----- ----------------------------------  3/24 [pybase64]
    Uninstalling protobuf-7.34.1:
   ----- --------------------


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## PDF 문서 로드

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/snow-white.pdf")
raw_docs = loader.load()

print("로드된 document 수 : ", len(raw_docs))

로드된 document 수 :  6


In [3]:
for i, doc in enumerate(raw_docs[:2],start=1):
    print("metadata :",doc.metadata)
    print("content preview :",doc.page_content[:200])
    print("-"*100)    

metadata : {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'data/snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
content preview : 백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.
----------------------------------------------------------------------------------------------------
metadata : {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'data/snow-white.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}
content preview : 왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 

## Text Splitter로 문서 청크 만들기

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_text = "대한민국 역사는 매우 길고 다양하다. 고조선부터 시작해서 삼국시대, 고려, 조선, 현대에 이르기까지 수 많은 사건과 인물이 존재한다."
sample_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 30,            # 하나의 chunk의 최대 길이
    chunk_overlap = 10          # 인접 chunk사이에 겹쳐 넣을 길이
)

sample_chunks = sample_splitter.split_text(sample_text)

for i,chunk in enumerate(sample_chunks, start=1):
    print(f"[Chunk {i}] {chunk}")

[Chunk 1] 대한민국 역사는 매우 길고 다양하다. 고조선부터
[Chunk 2] 고조선부터 시작해서 삼국시대, 고려, 조선, 현대에
[Chunk 3] 조선, 현대에 이르기까지 수 많은 사건과 인물이
[Chunk 4] 사건과 인물이 존재한다.


In [5]:
# PDF 문서를 읽어온 Document 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    add_start_index=True        # 원본 문서에 chunk가 시작 된 위치를 metadata에 저장
)

split_docs = text_splitter.split_documents(raw_docs)

print("원본 Document 수 :",len(raw_docs))
print("분할 후 Document 수 :",len(split_docs))

for i, doc in enumerate(split_docs[:5],start=1):
    print(f"\n[chunk] {i}")
    print("metadata :",doc.metadata)
    print(doc.page_content)

원본 Document 수 : 6
분할 후 Document 수 : 8

[chunk] 1
metadata : {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'data/snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'start_index': 0}
백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.

[chunk] 2
metadata : {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'data/snow-white.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2', 'start_index': 0}
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을

## Embedding과 Chroma vector store 생성

In [6]:
import os

DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory='./db/chroma',         # Chroma DB가 저장되는 폴더 경로
    collection_name='snow_white'             # 이 벡터 DB 안에서 이번 문서 묶음에 붙이는 이름
)

print("저장 된 문서 수 : ",vector_store._collection.count())

저장 된 문서 수 :  8


## Vector Score 직접 검색
- 답변 생성 전 검색 결과를 먼저 확인하는 습관이 중요하다.
- 검색 된 문서가 질문과 관련이 없다면 LLM 모델이 사용했을 때도 좋은 답변을 만들기 어렵다.

In [9]:
query = "왕비와 백설공주 중에 누가 더 아름답다고 했나요?"

# similarity_search_with_score : Document와 유사도 점수를 함께 반환한다.
# score는 거리 값으로 해석할 수 있으며 낮을 수록 가깝다.
search_results = vector_store.similarity_search_with_score(query,k=3)

for i, (doc,score) in enumerate(search_results,start=1):
    print(f"[검색 결과 {i}] score = {score}")
    print("metadata :", doc.metadata)
    print(doc.page_content)
    print("-"*80)

[검색 결과 1] score = 0.9356008172035217
metadata : {'creationdate': '2023-09-12T11:20:24+09:00', 'start_index': 0, 'moddate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'page_label': '4', 'creator': 'Microsoft® PowerPoint® 2013', 'producer': 'Microsoft® PowerPoint® 2013', 'total_pages': 6, 'page': 3, 'author': 'PC', 'source': 'data/snow-white.pdf'}
왕비는 먹음직스럽게 생긴 사과를 골라 독을 발랐어요.
그리고 과일 장수로 변장했지요.
왕비는 산을 넘고 또 넘어 일곱 난쟁이의 오두막에 도착했어요.
“새콤달콤 맛있는 사과가 있어요. 아가씨의 붉은 입술처럼 새빨
간 사과랍니다. 잠깐 문을 열어 보세요.”
백설공주는 고개를 저었어요.
“난쟁이들이 문을 열어 주지 말라고 했어요.”
백설공주가 거절하자, 왕비는 창문 틈새로 사과를 쑥 내밀었어
요.
“그럼, 맛이라도 봐요. 정말 맛있으니까. 둘이 먹다 하나가 죽어
도 모를걸요.”
“탐스러운 사과네. 맛있어 보여. 한입만 아삭 깨물어 볼까?”
사과를 베어 문 순간, 백설공주는 온몸에 독이 퍼져 정신을 잃고
쓰러졌어요.
“호호호. 이제 내가 세상에서 가장 아름답겠지?”
왕비는 백설공주를 버려둔 채 자리를 떠났어요.
--------------------------------------------------------------------------------
[검색 결과 2] score = 0.9382525682449341
metadata : {'total_pages': 6, 'start_index': 0, 'source': 'data/snow-white.pdf', 'author': 'PC', 'creator': 'Microsoft®

## Retriever 만들기
- 사용자의 질문을 받아 관련 Document 목록을 반환하는 표준 인터페이스

In [10]:
retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs={"k":3}
)

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs,start=1):
    print(f"[검색 결과 {i}] score = {score}")
    print("metadata :", doc.metadata)
    print(doc.page_content)
    print("-"*80)

[검색 결과 1] score = 1.0168721675872803
metadata : {'producer': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'page_label': '4', 'moddate': '2023-09-12T11:20:24+09:00', 'source': 'data/snow-white.pdf', 'start_index': 0, 'author': 'PC', 'title': 'PowerPoint 프레젠테이션', 'creator': 'Microsoft® PowerPoint® 2013', 'total_pages': 6, 'page': 3}
왕비는 먹음직스럽게 생긴 사과를 골라 독을 발랐어요.
그리고 과일 장수로 변장했지요.
왕비는 산을 넘고 또 넘어 일곱 난쟁이의 오두막에 도착했어요.
“새콤달콤 맛있는 사과가 있어요. 아가씨의 붉은 입술처럼 새빨
간 사과랍니다. 잠깐 문을 열어 보세요.”
백설공주는 고개를 저었어요.
“난쟁이들이 문을 열어 주지 말라고 했어요.”
백설공주가 거절하자, 왕비는 창문 틈새로 사과를 쑥 내밀었어
요.
“그럼, 맛이라도 봐요. 정말 맛있으니까. 둘이 먹다 하나가 죽어
도 모를걸요.”
“탐스러운 사과네. 맛있어 보여. 한입만 아삭 깨물어 볼까?”
사과를 베어 문 순간, 백설공주는 온몸에 독이 퍼져 정신을 잃고
쓰러졌어요.
“호호호. 이제 내가 세상에서 가장 아름답겠지?”
왕비는 백설공주를 버려둔 채 자리를 떠났어요.
--------------------------------------------------------------------------------
[검색 결과 2] score = 1.0168721675872803
metadata : {'creator': 'Microsoft® PowerPoint® 2013', 'title': 'PowerPoint 프레젠테이션', 'moddate': '2023-09-12T11:20:24+09:

## 검색 결과 Prompt에 넣기 좋은 문자열로 변환

In [11]:
def format_docs(docs:list) -> str:
    """검색된 Document 목록을 Prompt에 넣게 좋은 문자열로 변환한다."""

    formatted = []

    for i,doc in enumerate(docs,start=1):
        source = doc.metadata.get("source","unknown")
        page = doc.metadata.get("page","unknown")
        start_index = doc.metadata.get("start_index","unnown")

        formatted.append(
            f"""[문서 {i}]
source : {source}
page : {page}
start_index : {start_index}
content : {doc.page_content}
"""
        )
    return "\n\n".join(formatted)

docs_for_preview = retriever.invoke("왕자가 백설공주를 찾았을 때 백설공주는 어디에 있었나요?")
print(format_docs(docs_for_preview))

[문서 1]
source : data/snow-white.pdf
page : 5
start_index : 0
content : 왕자는 깨어난 백설공주를 보고 기뻐했어요.
“공주님, 나는 이웃 나라 왕자입니다.”
“왕자님이 나를 다시 살려 주셨군요.”
“나와 결혼해 주시겠어요?”
“네, 좋아요!”
두 사람은 일곱 난쟁이와 함께 오래오래 행복하게 살
았답니다.


[문서 2]
source : data/snow-white.pdf
page : 0
start_index : 0
content : 백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.


[문서 3]
source : data/snow-white.pdf
page : 4
start_index : 0
content : 저녁이 되자, 일곱 난쟁이가 돌아왔어요.
난쟁이들은 쓰러진 백설공주를 보고 엉엉 울었어요.
백설공주는 깊은 잠에 빠진 것처럼 보였지요.
“백설공주님, 못된 왕비의 꾐에 넘어갔군요.”
“여전히 아름다운 우리 공주님을 캄캄한 땅속에 묻을 순 없어.”
“오래오래 볼 수 있게 유리 관에 모시자.”
어느 날, 한 왕자가 숲을 지나다가 유리관을 보았어요.
“누구지? 이 아름다운 여인은?”
“백설공주랍니다.”
왕자는 백설공주에게 반해 유리관을 달라고 부탁했어요.
일곱 난쟁이는 백설공주를 잘 지킨다는 약속을 받고 유리관을
내주었지요.
그런데 신하들이 유리관을 옮기다 돌부리에 툭! 백설공주 목
에서 사과 조각이 툭! 
“우아, 공주님이 살아났어!”



## RAG Prompt 만들기

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 어린 학습자에게 이야기를 친절하게 설명하는 선생님입니다. "
        "반드시 제공된 context만 근거로 답변하세요. "
        "context에서 확인할 수 없는 내용은 추측하지 말고 모른다고 답변하세요."
    ),
    (
        "human",
        """
다음 context를 참고하여 사용자 질문에 답변하세요.

[사용자 질문]
{question}

[context]
{context}

[응답 형식]
답변:
참조문서:
- 문서 번호와 page 정보를 간단히 적으세요.
"""
    ),
])

# Prompt에 실제로 어떤 값이 들어가는지 먼저 확인한다.
preview_question = "왕자가 백설공주를 찾아왔을 때 백설공주는 어디에 있었나요?"
preview_docs = retriever.invoke(preview_question)

prompt_preview = prompt.invoke({
    "question": preview_question,
    "context": format_docs(preview_docs),
})

print(prompt_preview)

messages=[SystemMessage(content='당신은 어린 학습자에게 이야기를 친절하게 설명하는 선생님입니다. 반드시 제공된 context만 근거로 답변하세요. context에서 확인할 수 없는 내용은 추측하지 말고 모른다고 답변하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n다음 context를 참고하여 사용자 질문에 답변하세요.\n\n[사용자 질문]\n왕자가 백설공주를 찾아왔을 때 백설공주는 어디에 있었나요?\n\n[context]\n[문서 1]\nsource : data/snow-white.pdf\npage : 5\nstart_index : 0\ncontent : 왕자는 깨어난 백설공주를 보고 기뻐했어요.\n“공주님, 나는 이웃 나라 왕자입니다.”\n“왕자님이 나를 다시 살려 주셨군요.”\n“나와 결혼해 주시겠어요?”\n“네, 좋아요!”\n두 사람은 일곱 난쟁이와 함께 오래오래 행복하게 살\n았답니다.\n\n\n[문서 2]\nsource : data/snow-white.pdf\npage : 0\nstart_index : 0\ncontent : 백설공주\n옛날 어느 왕국에 공주님이 태어났어요.\n“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백\n설공주라고 불러야겠다.”\n왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.\n하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어\n요.\n\n\n[문서 3]\nsource : data/snow-white.pdf\npage : 4\nstart_index : 0\ncontent : 저녁이 되자, 일곱 난쟁이가 돌아왔어요.\n난쟁이들은 쓰러진 백설공주를 보고 엉엉 울었어요.\n백설공주는 깊은 잠에 빠진 것처럼 보였지요.\n“백설공주님, 못된 왕비의 꾐에 넘어갔군요.”\n“여전히 아름다운 우리 공주님을 캄캄한 땅속에 묻을 순 없어.”\n“오래오래 볼 수 있게 유리 관에 모시자.”\n어느 날, 한 왕자가 숲을 지나다가 유

## RAG Chain 구성
- LCEL로 RAG Chain을 구성한다.

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    model=DEFAULT_MODEL,
    temperature=0.2
)

rag_chain = (
    {
        "context" : retriever | format_docs,
        "question" : RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("왕자가 백설공주를 찾았을 때 백설공주는 어디에 있었나요?")
print(answer)

답변:  
왕자가 백설공주를 찾았을 때 백설공주는 유리관 안에 있었어요. 신하들이 유리관을 옮기다가 백설공주 목에서 사과 조각이 빠져 나오면서 백설공주가 깨어났답니다.

참조문서:  
- 문서 3, page 4  
- 문서 1, page 5


## LLM 단독 답변과 RAG 답변 비교

In [15]:
question = "왕자가 백설공주를 찾아왔을  때 백설공주는 어디에 있었나요?"


print(llm.invoke(question).content)

백설공주 이야기에 따르면, 왕자가 백설공주를 찾아왔을 때 백설공주는 난쟁이들의 집 안, 보통 침대나 침대 근처에 누워 있거나 잠들어 있는 상태에 있었습니다. 왕자가 그녀에게 키스하여 마법이 풀리고 깨어나는 장면이 대표적입니다.


## 문서의 있는 질문 테스트

In [16]:
questions = [
    "백설공주가 먹은 독사과의 색깔은 무엇인가요?",
    "일곱 난쟁이는 백설공주에게 무엇을 조심하라고 했나요?"
]

for question in questions:
    print("질문 :", question)
    print(rag_chain.invoke(question))
    print("-"*80)

질문 : 백설공주가 먹은 독사과의 색깔은 무엇인가요?
답변: 백설공주가 먹은 독사과의 색깔은 새빨간 색깔입니다.

참조문서:
- 문서 2, page 3
--------------------------------------------------------------------------------
질문 : 일곱 난쟁이는 백설공주에게 무엇을 조심하라고 했나요?
답변:  
일곱 난쟁이는 백설공주에게 "조심조심 또 조심. 낯선 사람에게는 문을 열어 주지 마세요."라고 했어요.

참조문서:  
- 문서 2, page 2
--------------------------------------------------------------------------------


## 문서에 없는 질문 텍스트

In [17]:
questions = [
    "백설공주를 살려준 사냥꾼은 그 후에 어떻게 되었나요?"
]

for question in questions:
    print("질문 :", question)
    print(rag_chain.invoke(question))
    print("-"*80)

질문 : 백설공주를 살려준 사냥꾼은 그 후에 어떻게 되었나요?
답변:  
백설공주를 살려준 사냥꾼은 차마 공주를 해칠 수 없어서, 왕비가 찾지 못하도록 멀리 떠나라고 말하며 백설공주를 숲으로 도망가게 했습니다. 그 후 사냥꾼이 어떻게 되었는지는 제공된 내용에 나와 있지 않습니다.

참조문서:  
- 문서 1, page 1
--------------------------------------------------------------------------------
